In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as f
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DoubleType
 
# необходимые библиотеки для интеграции Spark с Kafka и PostgreSQL
spark_jars_packages = ",".join(
        [
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
            "org.postgresql:postgresql:42.4.0",
        ]
    )
 
# создаём spark сессию с необходимыми библиотеками в spark_jars_packages для интеграции с Kafka и PostgreSQL
spark = SparkSession.builder \
    .appName("FinalProject") \
    .config("spark.sql.session.timeZone", "UTC") \
    .config("spark.jars.packages", spark_jars_packages) \
    .getOrCreate()
 
# читаем из топика Kafka сообщения с акциями от ресторанов 
transaction_df = (
    spark.read
    .format('kafka')
    .option('kafka.bootstrap.servers', 'rc1a-f179p9ij8hivfed1.mdb.yandexcloud.net:9091')
    .option('kafka.security.protocol', 'SASL_SSL')
    .option('kafka.sasl.jaas.config', 'org.apache.kafka.common.security.scram.ScramLoginModule required username="producer_consumer" password="1q2w3e4r5t";') \
    .option('kafka.sasl.mechanism', 'SCRAM-SHA-512')
    .option('kafka.ssl.ca.location', '/root/.ssh/YandexInternalRootCA.crt')
    .option('kafka.sasl.username', 'producer_comsumer')
    .option('kafka.sasl.password', '1q2w3e4r5t')
    #.option('kafka.ssl.truststore.location', '/usr/lib/jvm/java-1.17.0-openjdk-amd64/lib/security/cacerts')
    .option('subscribe', 'transaction-service-input')
    .load()
)
transaction_df.printSchema()
transaction_df.show()
# определяем схему входного сообщения для json
transaction_schema = StructType(
    [
        StructField("operation_id" , StringType(), True),
        StructField("account_number_from", IntegerType(), True),
        StructField("account_number_to" , IntegerType(), True),
        StructField("currency_code" , IntegerType(), True),
        StructField("country" , StringType(), True),
        StructField("status" , StringType(), True),
        StructField("transaction_type" , IntegerType(), True),
        StructField("amount" , DoubleType(), True),
        StructField("transaction_dt" , TimestampType(), True)
    ]
)
 
currency_schema = StructType(
    [
        StructField("date_update" , TimestampType(), True),
        StructField("currency_code", IntegerType(), True),
        StructField("currency_code_with" , IntegerType(), True),
        StructField("currency_code_div" , DoubleType(), True)
    ]
)
message_schema = StructType(
    [
        StructField("object_id" , StringType(), True),
        StructField("object_type", StringType(), True),
        StructField("sent_dttm" , TimestampType(), True),
        StructField("payload" , StringType(), True)
    ]
)
 
# десериализуем из value сообщения json и фильтруем по времени старта и окончания акции
transaction_stream_df = (
        transaction_df
        .withColumn('value', f.col('value').cast(StringType()))
        .withColumn('data', f.from_json(f.col('value'), message_schema))
        #.selectExpr('data.*')
        #.withWatermark('timestamp', '10 minutes')
    )
transaction_stream_df.show()
 
transaction_stream_df = (
        transaction_df
        .withColumn('value', f.col('value').cast(StringType()))
        .withColumn('object_id', f.col('object_id').cast(StringType()))
        .withColumn('object_type', f.col('object_type').cast(StringType()))
        .withColumn('sent_dttm', f.col('sent_dttm').cast(TimestampType()))
        .withColumn('payload', f.when(f.col('object_type') == 'TRANSACTION', 
                                      f.from_json(f.col('value'), transaction_schema))
                                .when(f.col('object_type') == 'CURRENCY', 
                                      f.from_json(f.col('value'), currency_schema))
                    )
        .withWatermark('timestamp', '5 minutes')
    )
 
# (
#     transaction_stream_df
#     .write
#     .outputMode("append")
#     .format('jdbc')
#     .option('url', 'rc1b-exusdl5kvwaak3bg.mdb.yandexcloud.net:6432/final')
#     .option('driver', 'org.postgresql.Driver')
#     .option('dbtable', 'stg.transaction_service')
#     .option('user', 'db_user')
#     .option('password', '1q2w3e4r5t')
#     .save()
#     )